# Import modules

In [ ]:
import pandas as pd

# Data

In [ ]:
sample_annot = pd.read_csv('pancancer_metabolomics_v.0.3.4/pancancer_metabolomics/data/MasterMapping_MetImmune_03_16_2022_release.csv')
sample_annot

In [ ]:
tpm_rna_files = []
for file_name in list(sample_annot.RNAFile.unique()):
  if file_name.endswith('.tpm.gene_symbol.csv'):
    tpm_rna_files.append(file_name)
tpm_rna_files

In [ ]:
import os

file_names_tpm = []
for file in os.listdir('pancancer_metabolomics_v.0.3.4/pancancer_metabolomics/data/transcriptomics_processed'):
  if file in tpm_rna_files:
    file_names_tpm.append(file)
file_names_tpm

In [ ]:
sample_annot_tpm = sample_annot[sample_annot.RNAFile.isin(file_names_tpm)]
sample_annot_tpm

In [ ]:
sample_annot_tpm['SampleID'] = ['Sample_' + str(i) for i in range(len(sample_annot_tpm))]
sample_annot_tpm

In [ ]:
merged_transcriptomics = pd.DataFrame()
for i in file_names_tpm:
  file_read = pd.read_csv(f'pancancer_metabolomics_v.0.3.4/pancancer_metabolomics/data/transcriptomics_processed/{i}')
  file_read = file_read.set_index('gene_symbol')
  file_read.index.name = None

  # Check for overlapping columns and only add new ones
  new_cols = [col for col in file_read.columns if col not in merged_transcriptomics.columns]
  merged_transcriptomics = pd.concat([merged_transcriptomics, file_read[new_cols]], axis='columns')

merged_transcriptomics

In [ ]:
sample_annot_tpm.Dataset.unique()

In [ ]:
file_names_metabolomics = []
for file in os.listdir('pancancer_metabolomics_v.0.3.4/pancancer_metabolomics/data/metabolomics_processed'):
  for dataset in list(sample_annot_tpm.Dataset.unique()):
    if file.endswith(f'{dataset}.xlsx'):
      file_names_metabolomics.append(file)
    else:
      pass
file_names_metabolomics

In [ ]:
file_len = []
for i in file_names_metabolomics:
  file_read = pd.read_excel(f'pancancer_metabolomics_v.0.3.4/pancancer_metabolomics/data/metabolomics_processed/{i}', sheet_name= 'data_imputed')
  file_len.append(len(file_read))

file_len

In [ ]:
merged_metabolomics = pd.DataFrame()
for i in file_names_metabolomics:
  file_read = pd.read_excel(f'pancancer_metabolomics_v.0.3.4/pancancer_metabolomics/data/metabolomics_processed/{i}', sheet_name= 'data_imputed')
  file_read = file_read.set_index('Unnamed: 0')
  file_read.index.name = None

  new_cols = [col for col in file_read.columns if col not in merged_metabolomics.columns]
  merged_metabolomics = pd.concat([merged_metabolomics, file_read[new_cols]], axis='columns')


merged_metabolomics

In [ ]:
merged_transcriptomics = merged_transcriptomics[sample_annot_tpm.RNAID]
merged_transcriptomics

In [ ]:
sample_annot_tpm_to_metabolomic = {}
for i in sample_annot_tpm.RNAID:
  metabid = sample_annot_tpm[sample_annot_tpm['RNAID'] == i].MetabID.values[0]
  sample_annot_tpm_to_metabolomic[i] = metabid
len(sample_annot_tpm_to_metabolomic)

In [ ]:
merged_transcriptomics.columns = [sample_annot_tpm_to_metabolomic[i] for i in merged_transcriptomics.columns]
merged_transcriptomics

In [ ]:
common_samples = []
for i in merged_metabolomics.columns:
  if i in merged_transcriptomics.columns:
    common_samples.append(i)
  else:
    pass
len(common_samples)

In [ ]:
merged_transcriptomics_common =  merged_transcriptomics[common_samples]
merged_transcriptomics_common

In [ ]:
merged_metabolomics_common =  merged_metabolomics[common_samples]
merged_metabolomics_common

In [ ]:
merged_transcriptomics_common.to_csv('transcriptomics_tpm.csv', index = True)

In [ ]:
merged_metabolomics_common.to_csv('metabolomics_log2.csv', index = True)

In [ ]:
met_genes = pd.read_csv('human_gem_associated_genes.csv')
met_genes

In [ ]:
tpm_met = merged_transcriptomics_common[merged_transcriptomics_common.index.isin(met_genes['Symbol'])]
tpm_met

In [ ]:
tpm_met.to_csv('transcriptomics_tpm_metabolism_related_genes.csv', index = True)